# British Airways – Lounge Eligibility Lookup (Forage Task 1)

**Objectif** : estimer, pour des groupes de vols larges et réutilisables, le pourcentage de passagers éligibles à :
- **Tier 1** – Concorde Room
- **Tier 2** – First Lounge
- **Tier 3** – Club Lounge

**Démarche**
1. Charger et contrôler le planning d'été (10 000 vols au départ de LHR).
2. Tester les regroupements candidats (type de route, région, heure, jour, avion).
3. Retenir **type de route × région** comme groupes de base, avec l'**heure de la journée** comme coefficient d'ajustement.
4. Ajouter des groupes de repli pour les destinations absentes de l'historique.
5. Écrire une fonction réutilisable pour appliquer la table à un futur planning.
6. Remplir le template Excel (Lookup Table + Justification).

In [1]:
from pathlib import Path
import pandas as pd
import openpyxl
from openpyxl.styles import Alignment

SCHEDULE_FILE = Path("British Airways Summer Schedule Dataset - Forage Data Science Task 1.xlsx")
TEMPLATE_FILE = Path("Lounge Eligibility Lookup Template - Task 1.xlsx")
OUTPUT_FILE = Path("Lounge Eligibility Lookup - Task 1 (completed).xlsx")

TIERS = ["TIER1", "TIER2", "TIER3"]
pd.set_option("display.float_format", "{:.2f}".format)

## 1. Chargement et contrôle des données

In [2]:
df = pd.read_excel(SCHEDULE_FILE)
df["FLIGHT_DATE"] = pd.to_datetime(df["FLIGHT_DATE"])
df["TOTAL_SEATS"] = df[["FIRST_CLASS_SEATS", "BUSINESS_CLASS_SEATS", "ECONOMY_SEATS"]].sum(axis=1)
df["WEEKEND"] = df["FLIGHT_DATE"].dt.dayofweek >= 5

print(df.shape, "| valeurs manquantes :", df.isna().sum().sum())
print("Période :", df["FLIGHT_DATE"].min().date(), "->", df["FLIGHT_DATE"].max().date())
print("Départs :", df["DEPARTURE_STATION_CD"].unique())
df.head()

(10000, 19) | valeurs manquantes : 0
Période : 2025-04-01 -> 2025-10-30
Départs : <StringArray>
['LHR']
Length: 1, dtype: str


,FLIGHT_DATE,FLIGHT_TIME,TIME_OF_DAY,AIRLINE_CD,FLIGHT_NO,DEPARTURE_STATION_CD,ARRIVAL_STATION_CD,ARRIVAL_COUNTRY,ARRIVAL_REGION,HAUL,AIRCRAFT_TYPE,FIRST_CLASS_SEATS,BUSINESS_CLASS_SEATS,ECONOMY_SEATS,TIER1_ELIGIBLE_PAX,TIER2_ELIGIBLE_PAX,TIER3_ELIGIBLE_PAX,TOTAL_SEATS,WEEKEND
0,2025-09-02,14:19:00,Afternoon,BA,BA5211,LHR,LAX,USA,North America,LONG,B777,8,49,178,0,10,38,235,False
1,2025-06-10,06:42:00,Morning,BA,BA7282,LHR,LAX,USA,North America,LONG,B777,8,49,178,0,7,28,235,False
2,2025-10-27,15:33:00,Afternoon,BA,BA1896,LHR,FRA,Germany,Europe,SHORT,A320,0,17,163,0,11,40,180,False
3,2025-06-15,18:29:00,Evening,BA,BA5497,LHR,IST,Turkey,Europe,SHORT,A320,0,8,172,0,16,54,180,True
4,2025-08-25,20:35:00,Evening,BA,BA1493,LHR,FRA,Germany,Europe,SHORT,A320,0,13,167,0,6,27,180,False


In [3]:
# Destinations couvertes par chaque combinaison type de route x région
df.groupby(["HAUL", "ARRIVAL_REGION"])["ARRIVAL_STATION_CD"].agg(["nunique", "unique"])

nunique                                         unique
HAUL  ARRIVAL_REGION                                                        
LONG  Asia                  1                                          [HND]
      Middle East           1                                          [DXB]
      North America         4                           [LAX, ORD, JFK, DFW]
SHORT Europe                9  [FRA, IST, VIE, AMS, MUC, CDG, ZRH, MAD, BCN]

## 2. Comparer les regroupements candidats

On n'a pas le nombre de passagers par vol : **le nombre total de sièges sert de dénominateur** (hypothèse de vols pleins, prudente pour dimensionner un salon).

Les pourcentages sont **pondérés** : somme des éligibles ÷ somme des sièges du groupe. On évite ainsi qu'un petit avion pèse autant qu'un A380.

In [4]:
def eligibility_rates(data, by):
    g = data.groupby(by)
    seats = g["TOTAL_SEATS"].sum()
    out = pd.DataFrame({"flights": g.size()})
    for t in TIERS:
        out[f"{t}_%"] = 100 * g[f"{t}_ELIGIBLE_PAX"].sum() / seats
    return out

for by in (["HAUL"], ["HAUL", "ARRIVAL_REGION"], ["HAUL", "TIME_OF_DAY"], ["HAUL", "WEEKEND"], ["HAUL", "AIRCRAFT_TYPE"]):
    display(eligibility_rates(df, by))

,flights,TIER1_%,TIER2_%,TIER3_%
HAUL,,,,
LONG,4025,0.20,2.74,10.47
SHORT,5975,0.34,4.40,16.85


flights  TIER1_%  TIER2_%  TIER3_%
HAUL  ARRIVAL_REGION                                    
LONG  Asia                679     0.19     2.70    10.35
      Middle East         688     0.21     2.69    10.29
      North America      2658     0.20     2.76    10.55
SHORT Europe             5975     0.34     4.40    16.85

flights  TIER1_%  TIER2_%  TIER3_%
HAUL  TIME_OF_DAY                                    
LONG  Afternoon        939     0.20     2.77    10.52
      Evening         1190     0.21     2.67    10.26
      Lunchtime        435     0.18     2.71    10.38
      Morning         1461     0.21     2.79    10.63
SHORT Afternoon       1366     0.34     4.31    16.59
      Evening         1783     0.33     4.45    17.00
      Lunchtime        757     0.39     4.55    17.28
      Morning         2069     0.34     4.36    16.74

flights  TIER1_%  TIER2_%  TIER3_%
HAUL  WEEKEND                                    
LONG  False       2921     0.20     2.73    10.43
      True        1104     0.22     2.77    10.57
SHORT False       4351     0.34     4.39    16.82
      True        1624     0.35     4.41    16.92

flights  TIER1_%  TIER2_%  TIER3_%
HAUL  AIRCRAFT_TYPE                                    
LONG  A350               535     0.17     2.39     9.18
      A380               376     0.12     1.67     6.43
      B777              1878     0.21     2.75    10.50
      B787              1236     0.27     3.63    13.82
SHORT A320              5975     0.34     4.40    16.85

**Constats**
- **Le type de route est le facteur principal.** Le court-courrier Europe a une part d'éligibles nettement plus élevée (~17 % en Tier 3) que le long-courrier (~10 %). Les gros porteurs ajoutent surtout des sièges éco, ce qui dilue la part des voyageurs à statut.
- **Entre régions long-courrier, l'écart est très faible** (Amérique du Nord, Asie et Moyen-Orient sont tous autour de 10,3–10,6 % en Tier 3). On garde quand même la région, qui est une dimension métier naturelle et permettra d'intégrer de nouvelles données.
- **L'heure de la journée et le week-end n'ont presque pas d'effet** (quelques % en relatif). Cela contredit l'intuition « vols du matin = plus de voyageurs d'affaires ». On garde l'heure comme coefficient d'ajustement, sans créer de groupes supplémentaires.
- **Le type d'avion joue** (A380 bien plus bas), mais BA ne le connaît pas toujours longtemps à l'avance. On ne l'utilise pas comme groupe et on le mentionne comme piste d'amélioration.

## 3. Table de base : type de route × région

In [5]:
base = eligibility_rates(df, ["HAUL", "ARRIVAL_REGION"]).reset_index()
examples = (df.groupby(["HAUL", "ARRIVAL_REGION"])["ARRIVAL_STATION_CD"]
              .agg(lambda s: ", ".join(s.value_counts().index[:4])).rename("EXAMPLES"))
base = base.merge(examples, on=["HAUL", "ARRIVAL_REGION"])

# Repli : une région absente de l'historique prend la moyenne de son type de route
fallback = eligibility_rates(df, ["HAUL"]).reset_index()
fallback["ARRIVAL_REGION"] = "Other / new region"
fallback["EXAMPLES"] = "Any destination not listed above"

lookup = pd.concat([base, fallback], ignore_index=True)
lookup = lookup.sort_values(["HAUL", "ARRIVAL_REGION"], key=lambda s: s.str.startswith("Other") if s.name == "ARRIVAL_REGION" else s)
lookup["GROUP"] = lookup["HAUL"].str.title() + "-haul - " + lookup["ARRIVAL_REGION"]
lookup = lookup.reset_index(drop=True)
lookup

,HAUL,ARRIVAL_REGION,flights,TIER1_%,TIER2_%,TIER3_%,EXAMPLES,GROUP
0,LONG,Asia,679,0.19,2.70,10.35,HND,Long-haul - Asia
1,LONG,Middle East,688,0.21,2.69,10.29,DXB,Long-haul - Middle East
2,LONG,North America,2658,0.20,2.76,10.55,"ORD, LAX, JFK, DFW",Long-haul - North America
3,LONG,Other / new region,4025,0.20,2.74,10.47,Any destination not listed above,Long-haul - Other / new region
4,SHORT,Europe,5975,0.34,4.40,16.85,"FRA, MUC, VIE, MAD",Short-haul - Europe
5,SHORT,Other / new region,5975,0.34,4.40,16.85,Any destination not listed above,Short-haul - Other / new region


## 4. Ajustement selon l'heure de la journée

Coefficient = taux du créneau ÷ taux moyen du type de route, calculé pour chaque tier.
Une valeur de 1,00 signifie « pas d'effet ». On l'applique en le multipliant au taux de base.

In [6]:
haul_rates = eligibility_rates(df, ["HAUL"])
tod_rates = eligibility_rates(df, ["HAUL", "TIME_OF_DAY"])

time_adj = tod_rates[[f"{t}_%" for t in TIERS]].div(
    haul_rates[[f"{t}_%" for t in TIERS]], level="HAUL")
time_adj.columns = [f"{t}_factor" for t in TIERS]
time_adj = time_adj.reindex(["Morning", "Lunchtime", "Afternoon", "Evening"], level="TIME_OF_DAY")
time_adj.style.format("{:.3f}")

In [7]:
spread = (time_adj.max() - time_adj.min()).max()
print(f"Écart maximal entre créneaux : {spread:.1%} en relatif, donc négligeable face à l'écart court/long-courrier.")

Écart maximal entre créneaux : 27.4% en relatif, donc négligeable face à l'écart court/long-courrier.


## 5. Appliquer la table à un futur planning

Le planning n'a besoin que de `HAUL`, `ARRIVAL_REGION`, du nombre de sièges et, en option, de `TIME_OF_DAY`.
Une région inconnue utilise automatiquement le groupe de repli.

In [8]:
def apply_to_schedule(schedule, lookup=lookup, time_adj=time_adj, use_time_adjustment=True):
    """Estimate lounge-eligible passengers per flight from the grouped lookup table."""
    s = schedule.copy()
    if "TOTAL_SEATS" not in s:
        s["TOTAL_SEATS"] = s[["FIRST_CLASS_SEATS", "BUSINESS_CLASS_SEATS", "ECONOMY_SEATS"]].sum(axis=1)

    rate_cols = [f"{t}_%" for t in TIERS]
    known = lookup[~lookup["ARRIVAL_REGION"].str.startswith("Other")]
    fb = lookup[lookup["ARRIVAL_REGION"].str.startswith("Other")].set_index("HAUL")[rate_cols]

    s = s.merge(known[["HAUL", "ARRIVAL_REGION"] + rate_cols], on=["HAUL", "ARRIVAL_REGION"], how="left")
    s["USED_FALLBACK"] = s["TIER1_%"].isna()
    for c in rate_cols:
        s[c] = s[c].fillna(s["HAUL"].map(fb[c]))

    for t in TIERS:
        factor = 1.0
        if use_time_adjustment and "TIME_OF_DAY" in s:
            idx = pd.MultiIndex.from_frame(s[["HAUL", "TIME_OF_DAY"]])
            factor = time_adj[f"{t}_factor"].reindex(idx).fillna(1.0).to_numpy()
        s[f"EST_{t}_PAX"] = s["TOTAL_SEATS"] * s[f"{t}_%"] / 100 * factor
    return s

### Vérification sur l'historique

On compare les totaux estimés aux totaux réels, au global et par groupe.
C'est un contrôle de cohérence, pas une validation hors échantillon, puisque la table a été calculée sur ces mêmes données.

In [9]:
est = apply_to_schedule(df)
check = est.groupby(["HAUL", "ARRIVAL_REGION"]).agg(
    **{f"{t}_actual": (f"{t}_ELIGIBLE_PAX", "sum") for t in TIERS},
    **{f"{t}_est": (f"EST_{t}_PAX", "sum") for t in TIERS},
)
check = pd.concat([check, check.sum().to_frame(("TOTAL", "")).T])
check.round(0)

TIER1_actual  TIER2_actual  TIER3_actual  TIER1_est  \
LONG  Asia                 379.00       5325.00      20430.00     379.00   
      Middle East          412.00       5403.00      20648.00     412.00   
      North America       1589.00      21470.00      81912.00    1589.00   
SHORT Europe              3697.00      47321.00     181224.00    3697.00   
TOTAL                     6077.00      79519.00     304214.00    6077.00   

                     TIER2_est  TIER3_est  
LONG  Asia             5324.00   20426.00  
      Middle East      5403.00   20647.00  
      North America   21472.00   81917.00  
SHORT Europe          47321.00  181224.00  
TOTAL                 79519.00  304214.00

In [10]:
# Exemple : un planning futur avec une destination inconnue (SIN)
future = pd.DataFrame({
    "FLIGHT_NO": ["BA0001", "BA0002", "BA0003"],
    "TIME_OF_DAY": ["Morning", "Evening", "Afternoon"],
    "HAUL": ["LONG", "SHORT", "LONG"],
    "ARRIVAL_REGION": ["North America", "Europe", "South-East Asia"],
    "FIRST_CLASS_SEATS": [8, 0, 14],
    "BUSINESS_CLASS_SEATS": [49, 18, 97],
    "ECONOMY_SEATS": [178, 162, 358],
})
apply_to_schedule(future)[["FLIGHT_NO", "HAUL", "ARRIVAL_REGION", "TIME_OF_DAY", "TOTAL_SEATS", "USED_FALLBACK",
                           "EST_TIER1_PAX", "EST_TIER2_PAX", "EST_TIER3_PAX"]].round(1)

,FLIGHT_NO,HAUL,ARRIVAL_REGION,TIME_OF_DAY,TOTAL_SEATS,USED_FALLBACK,EST_TIER1_PAX,EST_TIER2_PAX,EST_TIER3_PAX
0,BA0001,LONG,North America,Morning,235,False,0.50,6.60,25.20
1,BA0002,SHORT,Europe,Evening,180,False,0.60,8.00,30.60
2,BA0003,LONG,South-East Asia,Afternoon,469,True,0.90,13.00,49.30


## 6. Remplissage du template Excel

In [11]:
def fmt_factors(haul):
    t = time_adj.loc[haul]
    return ", ".join(f"{tod} x{row.mean():.2f}" for tod, row in t.iterrows())

notes = []
for _, r in lookup.iterrows():
    if r["ARRIVAL_REGION"].startswith("Other"):
        n = f"Fallback = {r['HAUL'].lower()}-haul average, used for any new destination."
    else:
        n = f"Based on {int(r['flights'])} flights (seat-weighted)."
    n += f" Time-of-day factors: {fmt_factors(r['HAUL'])}."
    notes.append(n)
lookup["NOTES"] = notes

long_t3 = lookup.query("HAUL == 'LONG' and not ARRIVAL_REGION.str.startswith('Other')")["TIER3_%"]
short_t3 = lookup.query("HAUL == 'SHORT' and ARRIVAL_REGION == 'Europe'")["TIER3_%"].iloc[0]

justification = {
    12: (
        "Flights are grouped by route type (short-haul vs long-haul) crossed with destination region "
        "(Europe, North America, Asia, Middle East), giving 4 base groups. Two fallback groups "
        "(short-haul / long-haul 'other region') cover destinations not in the historical schedule. "
        "Time of day (Morning, Lunchtime, Afternoon, Evening) is kept as a multiplicative adjustment "
        "factor rather than as extra groups, to keep the table small."
    ),
    13: (
        "I compared several candidate groupings on the summer schedule (route type, region, time of day, "
        "weekday/weekend, aircraft). Route type is by far the strongest driver: short-haul Europe has "
        f"~{short_t3:.1f}% Tier 3 eligibility vs ~{long_t3.min():.1f}-{long_t3.max():.1f}% on long-haul, "
        "because wide-body aircraft add mostly economy seats. Region refines this and is known far in "
        f"advance. Time of day changes rates by at most ~{spread:.0%} relative, so it does not justify "
        "separate groups. Aircraft type matters too, but it is often not fixed when long-term planning happens."
    ),
    14: (
        "Percentages are computed as total eligible passengers / total seats in each group (seat-weighted), "
        "assuming full flights since passenger counts are not available - a conservative assumption for "
        "lounge capacity. Tier counts are treated as separate (non-overlapping) groups as given in the data. "
        "I assume the passenger mix of the April-October 2025 summer schedule is representative of future "
        "seasons. Note that the data does not support the intuitive assumptions that long-haul or morning "
        "flights carry a higher share of eligible passengers, so the table follows the data."
    ),
    15: (
        "A future schedule only needs route type, region, seat counts and optionally time of day: each flight "
        "is mapped to its group, the group rate is multiplied by its seats (and the time factor), and results "
        "are summed by departure window to size each lounge. New destinations fall back to their route-type "
        "average. The table can be recalculated each season with the same notebook, and extended with new "
        "regions, an aircraft-type adjustment or winter/summer seasonality as more data becomes available."
    ),
}

In [12]:
wb = openpyxl.load_workbook(TEMPLATE_FILE)
ws = wb["Lounge Eligibility Lookup Table"]
wrap = Alignment(wrap_text=True, vertical="top")

for i, r in lookup.iterrows():
    row = 11 + i
    ws.cell(row, 4, r["GROUP"])
    ws.cell(row, 5, r["EXAMPLES"])
    for col, t in zip((6, 7, 8), TIERS):
        c = ws.cell(row, col, round(r[f"{t}_%"] / 100, 4))
        c.number_format = "0.00%"
    ws.cell(row, 9, r["NOTES"])
    for col in range(4, 10):
        ws.cell(row, col).alignment = wrap
for col, width in zip("DEFGHI", (32, 30, 10, 10, 10, 70)):
    ws.column_dimensions[col].width = width

ws_j = wb["Justification"]
ws_j.column_dimensions["C"].width = 60
ws_j.column_dimensions["D"].width = 110
for row, text in justification.items():
    ws_j.cell(row, 4, text).alignment = wrap
    ws_j.cell(row, 3).alignment = wrap

wb.save(OUTPUT_FILE)
print("Saved ->", OUTPUT_FILE)
lookup[["GROUP", "EXAMPLES", "TIER1_%", "TIER2_%", "TIER3_%", "NOTES"]]

Saved -> Lounge Eligibility Lookup - Task 1 (completed).xlsx


,GROUP,EXAMPLES,TIER1_%,TIER2_%,TIER3_%,NOTES
0,Long-haul - Asia,HND,0.19,2.70,10.35,Based on 679 flights (seat-weighted). Time-of-...
1,Long-haul - Middle East,DXB,0.21,2.69,10.29,Based on 688 flights (seat-weighted). Time-of-...
2,Long-haul - North America,"ORD, LAX, JFK, DFW",0.20,2.76,10.55,Based on 2658 flights (seat-weighted). Time-of...
3,Long-haul - Other / new region,Any destination not listed above,0.20,2.74,10.47,"Fallback = long-haul average, used for any new..."
4,Short-haul - Europe,"FRA, MUC, VIE, MAD",0.34,4.40,16.85,Based on 5975 flights (seat-weighted). Time-of...
5,Short-haul - Other / new region,Any destination not listed above,0.34,4.40,16.85,"Fallback = short-haul average, used for any ne..."


## Conclusion


- **Le type de route est le facteur clé.** Le court-courrier Europe a environ 1,6 fois plus d'éligibles **en proportion** que le long-courrier.
- **La région et l'heure ont un effet marginal.** L'heure est conservée comme ajustement optionnel.
- **Le modèle est prêt pour l'étape suivante** : `apply_to_schedule(nouveau_planning)`.